# Read Static Memory pointer from WASD

In [ ]:
import ctypes
import win32api
import win32process
import win32con
import keyboard
import time

PROCESS_ALL_ACCESS = (0x1F0FFF)

# Get the process ID of the target process (can be obtained using `psutil` or manually)
process_id = 3276  # replace with the actual process ID you want to target

# Open the target process with all access
process_handle = ctypes.windll.kernel32.OpenProcess(PROCESS_ALL_ACCESS, False, process_id)

if process_handle:
    print("Process opened successfully.")
    
    # Read key events in a loop
    while True:
        event = keyboard.read_event()

        # Check if the event is a key press
        if event.event_type == keyboard.KEY_DOWN:
            key = event.name.upper()  # Convert the key name to uppercase for comparison

            if key in ['W', 'A', 'S', 'D']:
                time.sleep(0.0013)
                # Define the memory address you want to read (example address)
                memory_address = 0x1E0E240CE08
                value = ctypes.c_uint32()

                # Read the memory from the target process
                bytes_read = ctypes.c_size_t()
                result = ctypes.windll.kernel32.ReadProcessMemory(
                    process_handle,
                    ctypes.c_void_p(memory_address),
                    ctypes.byref(value),
                    ctypes.sizeof(value),
                    ctypes.byref(bytes_read)
                )

                if result:
                    print(f"Value at address {hex(memory_address)}: {value.value}")
                else:
                    print("Failed to read memory.")
            
            # Exit the loop if the ESC key is pressed
            elif event.name == 'esc':
                print("Escape key pressed. Exiting...")
                break

    # Close the handle when the loop is done
    ctypes.windll.kernel32.CloseHandle(process_handle)
    print("Process handle closed.")
else:
    print("Failed to open process.")


Process opened successfully.
Value at address 0x1e0e240ce08: 9
Value at address 0x1e0e240ce08: 9
Value at address 0x1e0e240ce08: 9
Value at address 0x1e0e240ce08: 9
Value at address 0x1e0e240ce08: 9


# Try to read from base adres

In [36]:
import ctypes
import psutil

# now read the adres

# Constants
PROCESS_ALL_ACCESS = 0x1F0FFF

# Function to get the process ID
def get_process_id(process_name):
    for proc in psutil.process_iter(['pid', 'name']):
        if proc.info['name'] == process_name:
            return proc.info['pid']
    return None

# Function to read memory from a given process at a specific address
def read_memory(process_handle, address, data_type=ctypes.c_uint64):
    value = data_type()
    bytes_read = ctypes.c_size_t()
    result = ctypes.windll.kernel32.ReadProcessMemory(process_handle,ctypes.c_void_p(address),ctypes.byref(value),ctypes.sizeof(value),ctypes.byref(bytes_read))
    if result:
        return value.value
    else:
        error_code = ctypes.GetLastError()
        print(f"Failed to read memory. Error code: {error_code}")
        return None

# Main validation
process_name = "WASD.exe"
process_id = get_process_id(process_name)

# Open the process
process_handle = ctypes.windll.kernel32.OpenProcess(PROCESS_ALL_ACCESS, False, process_id)
if process_handle:
    # Base address of PsychEngine.exe
    base_address = 0x7FFB105B0000
    first_offset = 0x00487238
    first_pointer = base_address + first_offset
    dereferenced_address_1  = read_memory(process_handle, first_pointer)
    dereferenced_address_1_hex = (hex(dereferenced_address_1))
    print(dereferenced_address_1_hex)

    second_offset = 0x890
    second_pointer = dereferenced_address_1 + second_offset
    dereferenced_address_2  = read_memory(process_handle, second_pointer)
    dereferenced_address_2_hex = (hex(dereferenced_address_2))
    print(dereferenced_address_2_hex)

    third_offset = 0xF48
    third_pointer = dereferenced_address_2 + third_offset
    print(f'Address: {hex(third_pointer)}')
    dereferenced_address_3  = read_memory(process_handle, third_pointer)
    dereferenced_address_3_hex = (hex(dereferenced_address_3))
    print(f'value: {dereferenced_address_3}')

0x19c0e400b50
0x19c10c0bec0
Address: 0x19c10c0ce08
value: 10
